# Challenge 3: Robust Tagging Under Changing Detector Conditions

## What's the challenge?

A particle physics detector maps candidate particles in **η (eta) and φ (phi)** — pseudorapidity and azimuthal angle. When detector elements go offline — dead channels, noisy modules, alignment failures — particles whose trajectories pass through those regions simply don't show up in the event record.

Your task: **train a model whose anomaly detection stays robust as more of the detector drops out.**

At eval time, your model receives events where affected candidates are missing (represented as zeros, same as padding). How you train for this is entirely up to you — what augmentation strategy you use, what architecture, what objective. The only fixed constraints are the input format and what `predict()` outputs.

---

## Training data

- **Background only**: QCD, Drell-Yan, tt̄, W+jets — what the detector sees most of the time
- Each event: `[N, 7]` tensor of PF candidates with features `(pt, η, φ, dxy, dxy_sig, is_pf, pdgId)`. Padding rows (and missing candidates) have `pt == 0`.

## Scoring

At eval time your model sees both background and signal events and outputs a per-event anomaly score. Scores are evaluated via **AUC vs. degradation severity**. A robust model maintains high AUC as more of the detector goes dark. The leaderboard score is the area under that curve.

---

## How to use this notebook

1. **Edit `Model`** (Section 2) — implement `fit()` and `predict()`. The baseline is a working starting point; change anything.
2. **Run Train** (Section 3) and **Eval** (Section 4) to check performance.
3. **Submit** (Section 5) — paste your `Model` class into `model.py` and upload to Codabench.

The default is a working baseline.

---
## Section 1: Setup

In [ ]:
import glob
import os
import re

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

from embedding.models import TransformerEncoder, Projector
from embedding.preprocs import PFPreProcessor
from embedding.loss import SupConLoss
from embedding.training import make_train_val_split, build_train_val_loaders
from embedding.utils.data_utils import load_data

DATA_DIR = "REPLACE_ME"   # directory containing train + eval files
OUT_DIR  = "REPLACE_ME"   # where predictions are written

os.makedirs(OUT_DIR, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

---
## Section 2: Your `Model` class

`fit()` is where your full training strategy lives — architecture, augmentation, objective, all of it. `predict()` writes anomaly scores; its output format is fixed but everything that produces those scores is yours to design.

**The only constraints:**
- `predict()` outputs `[E, 2]` numpy arrays per eval file, column 1 = anomaly score in [0, 1]
- At eval time the model receives `[E, N, 7]` tensors where missing candidates are zeroed (same as padding — `pt == 0`)

**The baseline** uses a Transformer encoder trained with two objectives:
- **SupConLoss** — pairs each event with a degraded copy, pulling their embeddings together so the representation is stable under candidate dropout
- **CrossEntropyLoss** — keeps the embedding discriminative across the 4 background classes

The `_degrade` helper inside the baseline randomly kills a patch of candidates per event during training. You don't have to do it this way — augment differently, add extra input representations, use a different model or objective entirely. The baseline is a starting point, not a prescription.

---
## Section 3: Your `Model` class

`fit()` trains on background data. `predict()` writes per-event anomaly scores to `OUT_DIR` as `[E, 2]` arrays (column 1 = anomaly probability).

**The baseline architecture** is a Transformer that treats each event as a sequence of PF candidates:
- **PFPreProcessor** — normalizes the 7 raw input features per candidate
- **TransformerEncoder** — self-attention over the candidate sequence; a learned CLS token aggregates into a fixed-size event embedding
- **Projector** — maps the embedding to a smaller space for contrastive training
- **Classifier** — linear head over the 4 background classes (QCD, DY, tt̄, W+jets)

**Training** combines two objectives:
- **SupConLoss** — pulls together embeddings of the same event seen with and without degradation, so the representation doesn't collapse when the detector does
- **CrossEntropyLoss** on the nominal view — keeps the embedding discriminative between background types

At inference, anomaly score = `1 - max softmax(classifier(embedding))`. Events that don't look like any known background score high.

Feel free to change any of this — the architecture, the loss, or how you use `degrade`.

In [ ]:
# =====================================================================
# YOUR Model CLASS — change anything
# =====================================================================

NUM_BG_CLASSES = 4  # QCD, DY, TT, WJets

class Model:
    def __init__(self, data_dir, out_dir):
        self.data_dir = data_dir
        self.out_dir  = out_dir

        self.preproc    = PFPreProcessor(norm_constants={}).to(device)
        self.encoder    = TransformerEncoder(
            num_features=self.preproc.num_features,
            embed_size=128, latent_dim=6, num_heads=8, num_layers=4,
        ).to(device)
        self.projector  = Projector(6, 12, hidden_dim=48).to(device)
        self.classifier = nn.Linear(12, NUM_BG_CLASSES).to(device)

    @staticmethod
    def _degrade(x: torch.Tensor) -> torch.Tensor:
        """Baseline augmentation: kill one random eta-phi patch per event."""
        x = x.clone()
        B = x.size(0)
        eta_c = torch.empty(B, 1, device=x.device).uniform_(-2.0, 2.0)
        phi_c = torch.empty(B, 1, device=x.device).uniform_(-torch.pi, torch.pi)
        deta  = torch.empty(B, 1, device=x.device).uniform_(0.2, 1.5)
        dphi  = torch.empty(B, 1, device=x.device).uniform_(0.2, 1.5)
        eta, phi = x[..., 1], x[..., 2]
        dead = (
            (eta >= eta_c - deta / 2) & (eta < eta_c + deta / 2) &
            (phi >= phi_c - dphi / 2) & (phi < phi_c + dphi / 2) &
            (x[..., 0] > 0)
        )
        x[dead] = 0.0
        return x

    @staticmethod
    def _cls_mask(x):
        m = x[..., 0] == 0
        return torch.cat([torch.zeros(m.size(0), 1, device=m.device, dtype=torch.bool), m], dim=1)

    def _embed(self, x):
        return F.normalize(self.projector(self.encoder(self.preproc(x), None, self._cls_mask(x))), dim=1)

    def fit(self):
        feature_block, label_block = load_data(
            os.path.join(self.data_dir, "REPLACE_ME"),  # train file
            map_location="cpu",
        )
        X_tr, y_tr, X_val, y_val, _, _ = make_train_val_split(feature_block, label_block, val_size=0.1)
        train_loader, val_loader = build_train_val_loaders(
            X_tr, y_tr, X_val, y_val, device=device, batch_size=256, pfcands=True
        )

        criterion  = SupConLoss(temperature=0.07)
        ce_loss_fn = nn.CrossEntropyLoss()
        optimizer  = torch.optim.Adam(
            list(self.preproc.parameters()) + list(self.encoder.parameters())
            + list(self.projector.parameters()) + list(self.classifier.parameters()),
            lr=1e-3,
        )

        for epoch in range(10):
            self.preproc.train(); self.encoder.train()
            self.projector.train(); self.classifier.train()
            for x, _, labels in train_loader:
                x, labels = x.to(device), labels.to(device)
                x_aug = self._degrade(x)

                optimizer.zero_grad()
                emb_in  = self._embed(x)
                emb_aug = self._embed(x_aug)
                features = torch.stack([emb_in, emb_aug], dim=1)
                loss = criterion(features, labels) + ce_loss_fn(self.classifier(emb_in), labels)
                loss.backward()
                optimizer.step()

            self.preproc.eval(); self.encoder.eval()
            self.projector.eval(); self.classifier.eval()
            val_loss = val_correct = 0
            with torch.no_grad():
                for x, _, labels in val_loader:
                    x, labels = x.to(device), labels.to(device)
                    emb    = self._embed(x)
                    logits = self.classifier(emb)
                    val_loss    += ce_loss_fn(logits, labels).item() * x.size(0)
                    val_correct += (logits.argmax(1) == labels).float().sum().item()
            N = len(val_loader.dataset)
            print(f"epoch {epoch+1}/10  val_loss={val_loss/N:.4f}  val_acc={val_correct/N:.4f}")

    @torch.no_grad()
    def _anomaly_scores(self, path):
        features = torch.load(path, map_location=device)
        self.preproc.eval(); self.encoder.eval(); self.projector.eval(); self.classifier.eval()
        emb      = self._embed(features)
        bg_probs = torch.softmax(self.classifier(emb), dim=1)
        anomaly  = 1.0 - bg_probs.max(dim=1).values
        return torch.stack([1.0 - anomaly, anomaly], dim=1).cpu().numpy()

    def predict(self):
        np.save(os.path.join(self.out_dir, "pred_nominal.npy"),
                self._anomaly_scores(os.path.join(self.data_dir, "REPLACE_ME")))

        for path in glob.glob(os.path.join(self.data_dir, "REPLACE_ME", "*.pt")):
            sev = re.search(r"\d+", os.path.basename(path)).group()
            np.save(os.path.join(self.out_dir, f"pred_severity_{sev}.npy"), self._anomaly_scores(path))

# =====================================================================

---
## Section 3: Train

Runs `fit()` on the background training data.

In [ ]:
model = Model(DATA_DIR, OUT_DIR)
model.fit()

---
## Section 4: Evaluate

Runs `predict()` and plots AUC vs. severity. A flat curve near 1 is the goal.

In [ ]:
model.predict()

labels = np.load(os.path.join(DATA_DIR, "REPLACE_ME"))  # labels.npy (public eval)

def auc_from(pred_path):
    probs = np.load(pred_path)
    return roc_auc_score(labels, probs[:, 1])

severities = [0]
aucs = [auc_from(os.path.join(OUT_DIR, "pred_nominal.npy"))]

sev_files = sorted(
    glob.glob(os.path.join(OUT_DIR, "pred_severity_*.npy")),
    key=lambda p: int(re.search(r"\d+", os.path.basename(p)).group())
)
for path in sev_files:
    sev = int(re.search(r"\d+", os.path.basename(path)).group())
    severities.append(sev)
    aucs.append(auc_from(path))

robustness_score = np.trapz(aucs, severities) / (severities[-1] - severities[0]) if len(severities) > 1 else aucs[0]

print(f"\nRobustness score (area under AUC-vs-severity): {robustness_score:.4f}")
print(f"{'Severity':>10}  {'AUC':>8}")
for s, a in zip(severities, aucs):
    print(f"{s:>10}%  {a:>8.4f}")

plt.figure(figsize=(7, 5))
plt.plot(severities, aucs, marker="o", linewidth=2)
plt.axhline(0.5, color="gray", linestyle="--", alpha=0.5, label="random")
plt.xlabel("Dead detector fraction (% of η-φ plane)")
plt.ylabel("AUC (signal vs background)")
plt.title(f"Robustness score: {robustness_score:.4f}")
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

---
## Section 5: Submit to Codabench

Your submission is a single `model.py` file. Copy your `Model` class into it — the imports and `device` are already there.

Download `model.py` from the competition page, paste in your class, and upload. Codabench calls `Model(data_dir, out_dir).fit()` then `.predict()` and scores the output automatically.